# Recommender Demo: Sample Customer + Sample Item

Loads the real RetailRocket events, fits the `HybridRecommender`, and runs it
against one real customer (`visitorid`) and one real item (`itemid`) so you can
sanity-check recommendations end to end. Swap `SAMPLE_USER_ID` / `SAMPLE_ITEM_ID`
for any other id from `data/raw/events.csv` to try different cases.

**Tuned defaults now in effect:** event weights are view=1/addtocart=4/
transaction=10 (widened from 1/3/5), interactions are recency-decayed
(30-day half-life), `min_user_interactions`/`min_item_interactions` are
3/3 (down from 5/5), `min_known_interactions` is 2 (down from 3), and the
popularity fallback is category-scoped by default using the customer's
most recent item.

**Note on names:** RetailRocket is an anonymized dataset — there are no real
item names or customer names anywhere in it (item attributes are ~1000 hashed
numeric property codes; visitorid is a bare id with nothing attached). The one
real, non-hashed piece of item metadata is `categoryid`, so items below are
labeled with their category instead of a product name. Customers are labeled
as `Customer #<id>` since there's genuinely no other customer data to show.

In [1]:
import sys
sys.path.insert(0, '../src')

from data_loader import load_events, get_user_history
from recommender import HybridRecommender
from item_metadata import load_category_map, load_category_tree, item_label, customer_label

## 1. Load data and fit the model

In [2]:
events = load_events('../data/raw/events.csv')  # recency decay applied by default

rec = HybridRecommender(min_known_interactions=2, k_neighbors=20)
rec.fit(events, min_user_interactions=3, min_item_interactions=3)  # category map auto-loaded from cache

category_map = load_category_map()   # loads from data/processed/item_category_map.csv cache
category_tree = load_category_tree()

print(f'Events loaded: {len(events):,}')
print(f'Known items in CF model: {len(rec.item_cf.item_id_map):,}')
print(f'Known users in CF model: {len(rec.item_cf.user_id_map):,}')
print(f'Items with a known category: {len(category_map):,}')

Events loaded: 2,756,101
Known items in CF model: 43,858
Known users in CF model: 91,655
Items with a known category: 417,053


## 2. Pick a sample customer and sample item

`SAMPLE_USER_ID = 608628` is a real visitor with 4 distinct items in their
history (enough to clear the `min_known_interactions` threshold, so this one
actually exercises the item-based CF path rather than falling back to
popularity). `SAMPLE_ITEM_ID = 57245` is one of the items that customer bought.

In [3]:
SAMPLE_USER_ID = 608628
SAMPLE_ITEM_ID = 57245

history = get_user_history(events, SAMPLE_USER_ID)
print(f'{customer_label(SAMPLE_USER_ID)} — raw event log:')
print(events[events["visitorid"] == SAMPLE_USER_ID][["event", "itemid", "weight"]].to_string(index=False))
print(f'\nAggregated (recency-decayed) history used by the model:')
for itemid, weight in history:
    print(f'  {item_label(itemid, category_map, category_tree)}  weight={weight:.3f}')

Customer #608628 — raw event log:
      event  itemid  weight
       view  185598     1.0
       view  384528     1.0
transaction  185598    10.0
  addtocart   57245     4.0
       view  197754     1.0
transaction   57245    10.0
       view   57245     1.0
       view  197754     1.0
  addtocart  185598     4.0

Aggregated (recency-decayed) history used by the model:
  Item 57245 (Category 1393 -> 1383 -> 409 -> 140)  weight=15.000
  Item 185598 (Category 1393 -> 1383 -> 409 -> 140)  weight=15.000
  Item 197754 (Category 1393 -> 1383 -> 409 -> 140)  weight=2.000
  Item 384528 (Category 1393 -> 1383 -> 409 -> 140)  weight=1.000


## 3. Recommendations for this customer

In [4]:
recommendations = rec.recommend(user_id=SAMPLE_USER_ID, n=5)

print(f'Top 5 recommendations for {customer_label(SAMPLE_USER_ID)}:')
for itemid, score, source in recommendations:
    print(f'  {item_label(itemid, category_map, category_tree):<45}  score={score:.4f}  source={source}')

Top 5 recommendations for Customer #608628:
  Item 122411 (Category 1393 -> 1383 -> 409 -> 140)  score=0.2302  source=item_cf
  Item 245333 (Category 1393 -> 1383 -> 409 -> 140)  score=0.1581  source=item_cf
  Item 416992 (Category 1393 -> 1383 -> 409 -> 140)  score=0.1463  source=item_cf
  Item 181852 (Category 1393 -> 1383 -> 409 -> 140)  score=0.1318  source=item_cf
  Item 256134 (Category 1393 -> 1383 -> 409 -> 140)  score=0.1100  source=item_cf


## 4. Items similar to the sample item

This is the item-based CF layer in isolation: 'customers who interacted with
this item also interacted with...'.

In [5]:
similar = rec.item_cf.similar_items(SAMPLE_ITEM_ID, n=5)

print(f'Items most similar to {item_label(SAMPLE_ITEM_ID, category_map, category_tree)}:')
for itemid, similarity in similar:
    print(f'  {item_label(itemid, category_map, category_tree):<45}  similarity={similarity:.4f}')

Items most similar to Item 57245 (Category 1393 -> 1383 -> 409 -> 140):
  Item 122411 (Category 1393 -> 1383 -> 409 -> 140)  similarity=0.0621
  Item 185598 (Category 1393 -> 1383 -> 409 -> 140)  similarity=0.0492
  Item 416992 (Category 1393 -> 1383 -> 409 -> 140)  similarity=0.0395
  Item 245333 (Category 1393 -> 1383 -> 409 -> 140)  similarity=0.0393
  Item 256134 (Category 1393 -> 1383 -> 409 -> 140)  similarity=0.0266


## 5. Compare against a cold-start customer

Same call, but with an empty history — simulates a brand-new signup or a
new client with no data yet. Should fall back to popularity (global, since
there's no history to infer a category from).

In [6]:
cold_recommendations = rec.recommend(user_id=None, user_history=[], n=5)

print('Top 5 recommendations for a cold-start customer (no history):')
for itemid, score, source in cold_recommendations:
    print(f'  {item_label(itemid, category_map, category_tree):<45}  score={score:.4f}  source={source}')

Top 5 recommendations for a cold-start customer (no history):
  Item 461686 (Category 1037 -> 402 -> 143 -> 1482)  score=1978.5602  source=popularity_global
  Item 187946 (Category 1393 -> 1383 -> 409 -> 140)  score=1813.1765  source=popularity_global
  Item 320130 (Category 1483 -> 561 -> 395)      score=1034.1972  source=popularity_global
  Item 219512 (Category 5 -> 1637 -> 384 -> 140)  score=852.7231  source=popularity_global
  Item 384302 (Category 5 -> 1637 -> 384 -> 140)  score=816.4968  source=popularity_global
